# Stage 03: Data Transformation
Implements date decomposition, domain-specific imputation, encoding, feature selection, and train/test split per `flow.md`.

In [21]:
import os
if not os.getcwd().endswith("DataSciecne_Project_forcasting"):
    os.chdir("../")
%pwd

'/home/sheshpal/Documents/ml_llmops_project/Data_science[4]-end-to-end/DataSciecne_Project_forcasting'

In [22]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    reports_dir: Path

In [23]:
from src.forcastingproject.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.forcastingproject.utils.common import read_yaml, create_directories

In [24]:
class ConfigManager:
    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir, config.reports_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            reports_dir=Path(config.reports_dir),
        )
        return data_transformation_config

In [25]:
import os
import sys
import joblib
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from src.forcastingproject.logging.logger import logger
from src.forcastingproject.exception.exception import CustomException

In [26]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def _save_plot(self, name: str):
        os.makedirs(self.config.reports_dir, exist_ok=True)
        path = os.path.join(self.config.reports_dir, name)
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        logger.info(f"Saved plot: {path}")

    def _load_data(self) -> pd.DataFrame:
        data_path = self.config.data_path
        df = pd.read_csv(data_path)
        logger.info(f"Loaded data from {data_path} with shape {df.shape}")
        return df

    def _add_date_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df["Year"] = df["Date"].dt.year
        df["Month"] = df["Date"].dt.month
        df["Day"] = df["Date"].dt.day
        df["DayOfWeek"] = df["Date"].dt.dayofweek
        df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
        return df

    def explor_data_analysis(self):
        try:
            df = self._add_date_features(self._load_data())
            df["PromoInterval"] = df["PromoInterval"].fillna("")

            monthly_sales = df.groupby(["Year", "Month"], as_index=False)["Sales"].sum()
            monthly_sales["Date"] = pd.to_datetime(monthly_sales[["Year", "Month"]].assign(DAY=1))
            plt.figure(figsize=(14, 6))
            sns.lineplot(data=monthly_sales, x="Date", y="Sales")
            plt.title("Total Sales Over Time (Monthly)")
            plt.xlabel("Month")
            plt.ylabel("Sales")
            plt.xticks(rotation=45)
            plt.grid(True)
            plt.tight_layout()
            self._save_plot("monthly_sales_trend.png")

            top_stores = df.groupby("Store")["Sales"].sum().sort_values(ascending=False).head(10)
            plt.figure(figsize=(10, 5))
            sns.barplot(x=top_stores.index.astype(str), y=top_stores.values, palette="viridis")
            plt.title("Top 10 Stores by Total Sales")
            plt.xlabel("Store ID")
            plt.ylabel("Total Sales")
            plt.tight_layout()
            self._save_plot("top10_stores.png")

            dow_sales = df.groupby("DayOfWeek")["Sales"].mean().sort_index()
            plt.figure(figsize=(8, 5))
            sns.barplot(x=dow_sales.index, y=dow_sales.values, palette="magma")
            plt.title("Average Sales by Day of the Week (0=Mon)")
            plt.xlabel("Day of Week")
            plt.ylabel("Average Sales")
            plt.tight_layout()
            self._save_plot("sales_by_dayofweek.png")

            plt.figure(figsize=(6, 4))
            sns.barplot(x="Promo", y="Sales", data=df, palette="coolwarm")
            plt.title("Sales: Promo vs No Promo")
            plt.xlabel("Promo (0 = No, 1 = Yes)")
            plt.ylabel("Sales")
            plt.tight_layout()
            self._save_plot("promo_vs_sales.png")

            logger.info("EDA completed and plots saved")
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_data_transformation(self):
        try:
            df = self._add_date_features(self._load_data())

            # Domain-specific missing value imputation
            if "CompetitionDistance" in df.columns:
                df["CompetitionDistance"] = df["CompetitionDistance"].fillna(df["CompetitionDistance"].median())
            for col in ["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear", "Promo2SinceWeek", "Promo2SinceYear"]:
                if col in df.columns:
                    df[col] = df[col].fillna(0)
            if "PromoInterval" in df.columns:
                df["PromoInterval"] = df["PromoInterval"].fillna("None")

            # Feature selection: drop target and non-predictive/unavailable-at-forecast-time columns
            drop_cols = [c for c in ["Sales", "Date", "Customers"] if c in df.columns]
            target_col = "Sales"

            
            if target_col not in df.columns:
                raise ValueError("Target column 'Sales' not found in data")

            X = df.drop(columns=drop_cols)
            y = df[target_col]

            categorical_features = [c for c in ["StoreType", "Assortment", "StateHoliday", "PromoInterval"] if c in X.columns]
            numeric_features = [c for c in X.columns if c not in categorical_features]
            for col in categorical_features:
                X[col] = X[col].astype(str)


            numeric_transformer = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
            ])
            categorical_transformer = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ])

            preprocessor = ColumnTransformer(
                transformers=[
                    ("num", numeric_transformer, numeric_features),
                    ("cat", categorical_transformer, categorical_features),
                ],
                remainder="drop",
            )

            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

            X_train_arr = preprocessor.fit_transform(X_train)
            X_test_arr = preprocessor.transform(X_test)

            feature_names = preprocessor.get_feature_names_out()
            train_df = pd.DataFrame(X_train_arr, columns=feature_names)
            train_df["Sales"] = y_train.reset_index(drop=True).values
            test_df = pd.DataFrame(X_test_arr, columns=feature_names)
            test_df["Sales"] = y_test.reset_index(drop=True).values

            os.makedirs(self.config.root_dir, exist_ok=True)
            train_path = os.path.join(self.config.root_dir, "train.csv")
            test_path = os.path.join(self.config.root_dir, "test.csv")
            preprocessor_path = os.path.join(self.config.root_dir, "preprocessor.joblib")

            train_df.to_csv(train_path, index=False)
            test_df.to_csv(test_path, index=False)
            joblib.dump(preprocessor, preprocessor_path)

            logger.info(f"Saved train data to {train_path} with shape {train_df.shape}")
            logger.info(f"Saved test data to {test_path} with shape {test_df.shape}")
            logger.info(f"Saved preprocessor to {preprocessor_path}")
            print("Train shape:", train_df.shape)
            print("Test shape:", test_df.shape)
            print("Preprocessor saved to:", preprocessor_path)

            return train_path, test_path, preprocessor_path
        except Exception as e:
            raise CustomException(e, sys)

In [27]:
import traceback, datetime

try:
    config = ConfigManager()
    transform_config = config.get_data_transformation_config()
    transformer = DataTransformation(config=transform_config)
    transformer.explor_data_analysis()
    transformer.initiate_data_transformation()
except Exception as e:
    err_path = Path("artifacts/reports") / f"error_{datetime.datetime.now():%Y%m%d_%H%M%S}.log"
    err_path.parent.mkdir(parents=True, exist_ok=True)
    err_path.write_text(traceback.format_exc())
    print(f"Error report written to: {err_path}")
    raise e

[2026-11-15 20:09:54,123] forcastingProject INFO line 83: Created directory: artifacts
[2026-11-15 20:09:54,125] forcastingProject INFO line 83: Created directory: artifacts/data_transformation
[2026-11-15 20:09:54,126] forcastingProject INFO line 83: Created directory: artifacts/reports


/tmp/ipykernel_282237/771597712.py:14: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


[2026-11-15 20:09:56,286] forcastingProject INFO line 15: Loaded data from artifacts/data_ingestion/raw.csv with shape (1017209, 18)
[2026-11-15 20:09:57,678] forcastingProject INFO line 10: Saved plot: artifacts/reports/monthly_sales_trend.png
[2026-11-15 20:09:57,736] matplotlib.category INFO line 224: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2026-11-15 20:09:57,758] matplotlib.category INFO line 224: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


/tmp/ipykernel_282237/771597712.py:47: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_stores.index.astype(str), y=top_stores.values, palette="viridis")


[2026-11-15 20:09:58,666] forcastingProject INFO line 10: Saved plot: artifacts/reports/top10_stores.png
[2026-11-15 20:09:58,788] matplotlib.category INFO line 224: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2026-11-15 20:09:58,848] matplotlib.category INFO line 224: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


/tmp/ipykernel_282237/771597712.py:56: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=dow_sales.index, y=dow_sales.values, palette="magma")


[2026-11-15 20:09:59,530] forcastingProject INFO line 10: Saved plot: artifacts/reports/sales_by_dayofweek.png
[2026-11-15 20:10:00,574] matplotlib.category INFO line 224: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


/tmp/ipykernel_282237/771597712.py:64: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="Promo", y="Sales", data=df, palette="coolwarm")


[2026-11-15 20:10:01,263] matplotlib.category INFO line 224: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2026-11-15 20:10:19,351] forcastingProject INFO line 10: Saved plot: artifacts/reports/promo_vs_sales.png
[2026-11-15 20:10:19,359] forcastingProject INFO line 71: EDA completed and plots saved


/tmp/ipykernel_282237/771597712.py:14: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


[2026-11-15 20:10:21,202] forcastingProject INFO line 15: Loaded data from artifacts/data_ingestion/raw.csv with shape (1017209, 18)
[2026-11-15 20:11:10,653] forcastingProject INFO line 141: Saved train data to artifacts/data_transformation/train.csv with shape (813767, 31)
[2026-11-15 20:11:10,655] forcastingProject INFO line 142: Saved test data to artifacts/data_transformation/test.csv with shape (203442, 31)
[2026-11-15 20:11:10,656] forcastingProject INFO line 143: Saved preprocessor to artifacts/data_transformation/preprocessor.joblib
Train shape: (813767, 31)
Test shape: (203442, 31)
Preprocessor saved to: artifacts/data_transformation/preprocessor.joblib
